# Assignment 4: Advanced Image Generation with Stable Diffusion & ControlNet
**Course**: Research - Ada Lovelace  
**Hardware**: Optimized for NVIDIA A100 (Google Colab)

This notebook provides a robust pipeline for generating images with precise structural control using HuggingFace `diffusers`.

### 1. Installation & Setup
We install the latest `diffusers` and optimization libraries.

In [ ]:
#@title Setup Environment
!pip install -q diffusers transformers accelerate opencv-python controlnet_aux

import torch
import cv2
import numpy as np
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image
from transformers import pipeline as tf_pipeline

print(f"Using device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

### 2. Configuration & Model Loading
Set your parameters below. These will be rendered as inputs in Colab.

In [ ]:
#@title Model Configuration
base_model = "runwayml/stable-diffusion-v1-5" #@param ["runwayml/stable-diffusion-v1-5", "stabilityai/stable-diffusion-2-1"]
control_type = "canny" #@param ["canny", "depth"]

controlnet_map = {
    "canny": "lllyasviel/sd-controlnet-canny",
    "depth": "lllyasviel/sd-controlnet-depth"
}

print(f"Loading {control_type} ControlNet...")
controlnet = ControlNetModel.from_pretrained(controlnet_map[control_type], torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    base_model, controlnet=controlnet, torch_dtype=torch.float16
).to("cuda")

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_xformers_memory_efficient_attention()
print("Pipeline Ready!")

### 3. Image Preprocessing
Provide a source image URL to extract structure.

In [ ]:
#@title Source Image Selection
source_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/controlnet_training/canny_conditioning.png" #@param {type:"string"}
source_img = load_image(source_url)

if control_type == "canny":
    image = np.array(source_img)
    image = cv2.Canny(image, 100, 200)
    image = image[:, :, None]
    image = np.concatenate([image, image, image], axis=2)
    conditioning_image = Image.fromarray(image)
else:
    depth_estimator = tf_pipeline("depth-estimation")
    depth_map = depth_estimator(source_img)['depth']
    depth_map = np.array(depth_map)
    depth_map = depth_map[:, :, None]
    depth_map = np.concatenate([depth_map, depth_map, depth_map], axis=2)
    conditioning_image = Image.fromarray(depth_map)

print("Conditioning Image Extracted:")
conditioning_image.resize((256, 256))

### 4. Generation
Enter your prompt and generate a batch of images.

In [ ]:
#@title Inference Settings
prompt = "A futuristic library with neon glowing books, architectural photography, hyper-detailed, 8k" #@param {type:"string"}
negative_prompt = "blurry, low quality, distorted, watermark" #@param {type:"string"}
num_samples = 2 #@param {type:"slider", min:1, max:4, step:1}
steps = 30 #@param {type:"slider", min:10, max:100, step:5}
guidance_scale = 7.5 #@param {type:"number"}

generator = torch.Generator(device="cuda").manual_seed(42)

outputs = pipe(
    prompt=[prompt] * num_samples,
    negative_prompt=[negative_prompt] * num_samples,
    image=conditioning_image,
    num_inference_steps=steps,
    guidance_scale=guidance_scale,
    generator=generator
).images

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, num_samples, figsize=(20, 10))
for i in range(num_samples):
    axes[i].imshow(outputs[i])
    axes[i].axis('off')
plt.show()